# 01 · Extract → Bronze → Clean → Silver

**Pipeline step 1 of the Weather End-to-End project**

| Layer | Table | What it holds |
|---|---|---|
| Bronze | `bronze.weather_raw` | Raw API data, exactly as received |
| Silver | `silver.weather_daily` | Cleaned, typed, enriched data |

**Before running:** start SQL Server with `docker compose up -d sqlserver` and wait ~30 seconds.

Data source: [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) — free, no API key needed.

In [ ]:
import time
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", 30)

## 1. Database connection
Same settings as `docker-compose.yml`. If you changed the password there, change it here too.

In [ ]:
DB_SERVER   = "localhost"
DB_PORT     = 1433
DB_USER     = "sa"
DB_PASSWORD = "Weather_Pass123!"
DB_NAME     = "WeatherDB"

def get_engine(database="master"):
    url = URL.create("mssql+pymssql", username=DB_USER, password=DB_PASSWORD,
                     host=DB_SERVER, port=DB_PORT, database=database)
    return create_engine(url)

# quick connection test
with get_engine().connect() as conn:
    print(conn.execute(text("SELECT @@VERSION")).scalar()[:60])

## 2. Create the database + bronze / silver / gold schemas
Runs `sql/01_setup_database.sql`. `CREATE DATABASE` needs autocommit mode.

In [ ]:
def run_sql_file(path, engine, autocommit=False):
    """Run a .sql file that uses GO as batch separator."""
    sql = Path(path).read_text(encoding="utf-8")
    batches = [b.strip() for b in sql.split("\nGO") if b.strip()]
    conn = engine.connect()
    if autocommit:
        conn = conn.execution_options(isolation_level="AUTOCOMMIT")
    with conn:
        for batch in batches:
            if batch.upper().startswith("USE "):      # we connect to the right DB instead
                continue
            conn.execute(text(batch))
        if not autocommit:
            conn.commit()

master = get_engine("master")
with master.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("IF DB_ID('WeatherDB') IS NULL CREATE DATABASE WeatherDB"))

engine = get_engine(DB_NAME)
run_sql_file("../sql/01_setup_database.sql", engine, autocommit=True)

with engine.connect() as conn:
    print(pd.read_sql("SELECT name FROM sys.schemas WHERE name IN ('bronze','silver','gold')", conn))

## 3. Extract — get weather data from the API
Daily data for 5 cities, from 2022-01-01 until about a week ago (the archive has a few days' delay).

In [ ]:
CITIES = {
    "Baku":     {"country": "Azerbaijan", "lat": 40.41, "lon": 49.87},
    "London":   {"country": "UK",         "lat": 51.51, "lon": -0.13},
    "Berlin":   {"country": "Germany",    "lat": 52.52, "lon": 13.41},
    "New York": {"country": "USA",        "lat": 40.71, "lon": -74.01},
    "Dubai":    {"country": "UAE",        "lat": 25.20, "lon": 55.27},
}

START_DATE = "2022-01-01"
END_DATE   = (date.today() - timedelta(days=7)).isoformat()

DAILY_VARS = ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
              "precipitation_sum", "wind_speed_10m_max", "sunshine_duration"]

API_URL = "https://archive-api.open-meteo.com/v1/archive"


def fetch_city(city, info):
    params = {
        "latitude": info["lat"], "longitude": info["lon"],
        "start_date": START_DATE, "end_date": END_DATE,
        "daily": ",".join(DAILY_VARS), "timezone": "auto",
    }
    r = requests.get(API_URL, params=params, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["daily"])          # columns: time + the daily variables
    df["city"], df["country"] = city, info["country"]
    df["latitude"], df["longitude"] = info["lat"], info["lon"]
    return df


frames = []
for city, info in CITIES.items():
    print(f"Downloading {city} ...")
    frames.append(fetch_city(city, info))
    time.sleep(1)                                   # be polite to the free API

raw_df = pd.concat(frames, ignore_index=True)
print(raw_df.shape)
raw_df.head()

## 4. Load → BRONZE (raw, unchanged + load metadata)

In [ ]:
bronze_df = raw_df.copy()
bronze_df["source"] = "open-meteo-archive"
bronze_df["ingested_at"] = pd.Timestamp.now()

bronze_df.to_sql("weather_raw", engine, schema="bronze", if_exists="replace",
                 index=False, chunksize=100, method="multi")

with engine.connect() as conn:
    print("bronze rows:", conn.execute(text("SELECT COUNT(*) FROM bronze.weather_raw")).scalar())

## 5. Explore the raw data (read it back from bronze)

In [ ]:
bronze = pd.read_sql("SELECT * FROM bronze.weather_raw", engine)

print("Data types:\n", bronze.dtypes, "\n")
print("Missing values per column:\n", bronze.isna().sum(), "\n")
print("Duplicate city+date rows:", bronze.duplicated(["city", "time"]).sum())
bronze.describe()

## 6. Clean → SILVER
Cleaning steps:
1. Rename columns to clear names
2. Fix data types (text date → real date)
3. Remove duplicates (same city + date)
4. Drop rows where *all* measurements are missing
5. Fill small gaps by interpolation (per city, in date order)
6. Remove impossible values (e.g. negative rain, temperature > 60 °C)
7. Convert sunshine seconds → hours
8. Add useful columns: `temp_range`, `year`, `month`, `season`, `is_rainy`

In [ ]:
df = bronze.copy()

# 1. rename
df = df.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "temperature_2m_mean": "temp_mean",
    "precipitation_sum": "precipitation_mm",
    "wind_speed_10m_max": "wind_speed_max_kmh",
    "sunshine_duration": "sunshine_sec",
})

# 2. types
df["date"] = pd.to_datetime(df["date"], errors="coerce")
measures = ["temp_max", "temp_min", "temp_mean", "precipitation_mm", "wind_speed_max_kmh", "sunshine_sec"]
df[measures] = df[measures].apply(pd.to_numeric, errors="coerce")
df["city"] = df["city"].str.strip()

# 3. duplicates + rows without a date
before = len(df)
df = df.dropna(subset=["date"]).drop_duplicates(subset=["city", "date"], keep="last")
print("removed duplicates / bad dates:", before - len(df))

# 4. rows with no measurements at all
before = len(df)
df = df.dropna(subset=measures, how="all")
print("removed empty rows:", before - len(df))

# 6. impossible values -> NaN (then filled in step 5)
df.loc[~df["temp_max"].between(-90, 60), "temp_max"] = np.nan
df.loc[~df["temp_min"].between(-90, 60), "temp_min"] = np.nan
df.loc[~df["temp_mean"].between(-90, 60), "temp_mean"] = np.nan
df.loc[df["precipitation_mm"] < 0, "precipitation_mm"] = np.nan
df.loc[df["wind_speed_max_kmh"] < 0, "wind_speed_max_kmh"] = np.nan
df.loc[~df["sunshine_sec"].between(0, 86400), "sunshine_sec"] = np.nan

# 5. fill gaps per city (linear in time), leftover edges with the city median
df = df.sort_values(["city", "date"]).reset_index(drop=True)
df[measures] = df.groupby("city")[measures].transform(lambda s: s.interpolate(limit_direction="both"))
df[measures] = df[measures].fillna(df.groupby("city")[measures].transform("median"))
df["precipitation_mm"] = df["precipitation_mm"].fillna(0)

# 7. sunshine to hours
df["sunshine_hours"] = (df["sunshine_sec"] / 3600).round(2)
df = df.drop(columns="sunshine_sec")

# 8. new columns
df["temp_range"] = (df["temp_max"] - df["temp_min"]).round(2)
df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month
df["season"] = df["month"].map({12: "Winter", 1: "Winter", 2: "Winter",
                                3: "Spring", 4: "Spring", 5: "Spring",
                                6: "Summer", 7: "Summer", 8: "Summer",
                                9: "Autumn", 10: "Autumn", 11: "Autumn"})
df["is_rainy"] = (df["precipitation_mm"] >= 1.0).astype(int)
df["cleaned_at"] = pd.Timestamp.now()

silver_df = df[["city", "country", "latitude", "longitude", "date", "year", "month", "season",
                "temp_max", "temp_min", "temp_mean", "temp_range", "precipitation_mm",
                "wind_speed_max_kmh", "sunshine_hours", "is_rainy", "cleaned_at"]]
silver_df.head()

### Data-quality checks (the notebook stops here if something is wrong)

In [ ]:
assert silver_df[["city", "date"]].duplicated().sum() == 0, "duplicates found"
assert silver_df.isna().sum().sum() == 0, "missing values left"
assert (silver_df["temp_min"] <= silver_df["temp_max"] + 0.01).mean() > 0.99, "min > max too often"
assert (silver_df["precipitation_mm"] >= 0).all(), "negative rain"
print("All checks passed ✔  rows:", len(silver_df))
silver_df.groupby("city").size()

## 7. Load → SILVER

In [ ]:
silver_df.to_sql("weather_daily", engine, schema="silver", if_exists="replace",
                 index=False, chunksize=100, method="multi")

with engine.connect() as conn:
    print(pd.read_sql("""
        SELECT city, COUNT(*) AS days, MIN([date]) AS first_day, MAX([date]) AS last_day,
               ROUND(AVG(temp_mean), 1) AS avg_temp
        FROM silver.weather_daily
        GROUP BY city
    """, conn))

✅ **Done.** Continue with `02_gold_ml_analysis.ipynb`.